# Availability Test — Visualización

Visualiza los 3 escenarios de test del módulo `src/availability` **sin necesidad de API ni OSRM**.
Los datos provienen directamente de los builders sintéticos en `tests/availability/builders.py`.

| Color | Significado |
|-------|-------------|
| 🔵 Azul | Labor preassigned |
| 🟠 Naranja | Movimiento del conductor |
| 🟢 Verde | Slot factible |
| 🔴 Rojo | Slot no factible |
| ─ ─ Rojo | Desired slot solicitado |
| ─ ─ Verde | Primer slot factible disponible |

In [1]:
import sys
from pathlib import Path


import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

print(f"Project root: {_PROJECT_ROOT}")

Project root: /Users/jbeta/Documents/AlfredEnvs/AlfredDEV


In [2]:
from alfred.availability.slot_scanner import scan_availability
from tests.availability.builders import (
    build_feasible_desired_slot,
    build_infeasible_desired_slot_with_alternatives,
    build_no_drivers,
)

SCENARIOS = {
    "1 — Desired slot feasible (empty schedule)": build_feasible_desired_slot,
    "2 — Desired slot blocked (alternatives exist)": build_infeasible_desired_slot_with_alternatives,
    "3 — No drivers available": build_no_drivers,
}

_results = {}
for name, builder in SCENARIOS.items():
    state, request = builder()
    response = scan_availability(request, state)
    _results[name] = (state, request, response)
    feasible = response.desired_slot_result.feasible
    n_alts = len(response.feasible_slots)
    print(f"{name}")
    print(f"  desired_feasible={feasible}  scan_performed={response.scan_performed}  alternatives={n_alts}")

1 — Desired slot feasible (empty schedule)
  desired_feasible=True  scan_performed=False  alternatives=0
2 — Desired slot blocked (alternatives exist)
  desired_feasible=False  scan_performed=True  alternatives=17
3 — No drivers available
  desired_feasible=False  scan_performed=True  alternatives=0


In [3]:
from collections import defaultdict
import math

_BOGOTA_TZ = "America/Bogota"
_SLOT_DURATION_MIN = 30

# ── Colors ────────────────────────────────────────────────────────────────────
_C_PREASSIGNED_LABOR = "rgba( 55, 115, 200, 0.88)"
_C_PREASSIGNED_MOVE  = "rgba(250, 155,  45, 0.85)"
_C_SLOT_FEASIBLE     = "rgba( 40, 170,  80, 0.85)"
_C_SLOT_INFEASIBLE   = "rgba(220,  60,  60, 0.75)"


def _to_bogota_naive(ts) -> pd.Timestamp:
    """Convert any timestamp to a tz-naive Bogotá local time."""
    ts = pd.Timestamp(ts)
    if ts.tzinfo is not None:
        ts = ts.tz_convert(_BOGOTA_TZ).tz_localize(None)
    return ts


def _to_plotly_dt(ts) -> str:
    """Naive ISO string for bar base values and shape x positions."""
    return _to_bogota_naive(ts).isoformat()


def _add_vline(fig, ts, color, dash, label, annotation_side="right"):
    """
    add_vline with annotation_position is broken in Plotly 6.x for string x values.
    Use add_shape + add_annotation directly instead.
    """
    x = _to_plotly_dt(ts)
    fig.add_shape(
        type="line",
        x0=x, x1=x, y0=0, y1=1,
        xref="x", yref="paper",
        line=dict(color=color, dash=dash, width=2),
    )
    xanchor = "left" if annotation_side == "right" else "right"
    fig.add_annotation(
        x=x, y=1,
        xref="x", yref="paper",
        text=label,
        showarrow=False,
        xanchor=xanchor,
        yanchor="bottom",
        font=dict(color=color, size=11),
    )


# ── Summary card ──────────────────────────────────────────────────────────────
def build_summary_card(request, response) -> pd.DataFrame:
    ds = response.desired_slot_result
    slot_str = _to_bogota_naive(request.desired_slot).strftime("%H:%M")
    return pd.DataFrame([{
        "service_id":       request.service_id,
        "department":       request.department_code,
        "desired_slot":     slot_str,
        "desired_feasible": ds.feasible,
        "reason":           ds.reason or "—",
        "scan_performed":   response.scan_performed,
        "slots_checked":    response.total_slots_checked,
        "feasible_count":   len(response.feasible_slots),
    }]).set_index("service_id")


# ── Shared x-axis config ──────────────────────────────────────────────────────
def _xaxis_config(day_str) -> dict:
    x_min = _to_plotly_dt(pd.Timestamp(f"{day_str}T07:30:00").tz_localize(_BOGOTA_TZ))
    x_max = _to_plotly_dt(pd.Timestamp(f"{day_str}T19:30:00").tz_localize(_BOGOTA_TZ))
    return dict(type="date", tickformat="%H:%M", title_text="Time (Bogotá)", range=[x_min, x_max])


# ── Gantt ─────────────────────────────────────────────────────────────────────
def build_gantt_figure(state, request, response) -> go.Figure:
    labors_df = state.base_labors_df
    moves_df  = state.base_moves_df
    traces = []

    def _add_bars(df, color, name):
        if df is None or df.empty:
            return
        df = df.dropna(subset=["assigned_driver", "actual_start", "actual_end"]).copy()
        if df.empty:
            return
        base_vals, x_vals, y_vals, hovers = [], [], [], []
        for _, row in df.iterrows():
            start, end = row["actual_start"], row["actual_end"]
            dur_ms = (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds() * 1000
            if dur_ms <= 0:
                dur_ms = 5 * 60 * 1000
            base_vals.append(_to_plotly_dt(start))
            x_vals.append(dur_ms)
            y_vals.append(str(row["assigned_driver"]))
            ht = (
                f"<b>{name}</b><br>"
                f"Driver: {row['assigned_driver']}<br>"
                f"Service: {row.get('service_id', '?')}<br>"
                f"Start: {_to_bogota_naive(start).strftime('%H:%M')}<br>"
                f"End:   {_to_bogota_naive(end).strftime('%H:%M')}<br>"
                f"Duration: {dur_ms/60000:.0f} min"
            )
            hovers.append(ht)
        traces.append(go.Bar(
            x=x_vals, y=y_vals, base=base_vals, orientation="h",
            name=name, marker_color=color,
            hovertext=hovers, hoverinfo="text",
            legendgroup=name,
        ))

    _add_bars(labors_df, _C_PREASSIGNED_LABOR, "Labor preassigned")
    if moves_df is not None and not moves_df.empty and "labor_name" in moves_df.columns:
        _moves = moves_df[moves_df["labor_name"].astype(str).str.upper() == "DRIVER_MOVE"]
        _add_bars(_moves, _C_PREASSIGNED_MOVE, "Driver move")

    fig = go.Figure(data=traces)

    _add_vline(fig, request.desired_slot, "crimson", "dash",
               f"Desired {_to_bogota_naive(request.desired_slot).strftime('%H:%M')}",
               annotation_side="right")

    if response.feasible_slots:
        first = response.feasible_slots[0].slot_time
        _add_vline(fig, first, "green", "dot",
                   f"1st available {_to_bogota_naive(first).strftime('%H:%M')}",
                   annotation_side="left")

    drivers = (
        state.master_data.directorio_df["driver_id"].tolist()
        if not state.master_data.directorio_df.empty
        else ["(no drivers)"]
    )
    fig.update_layout(
        title_text="Preassigned Schedule",
        barmode="overlay",
        height=max(300, 80 * len(drivers) + 160),
        xaxis=_xaxis_config(state.day_str),
        yaxis=dict(title_text="Driver", categoryorder="array", categoryarray=list(reversed(drivers))),
        legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="center", x=0.5),
        hovermode="closest",
        margin=dict(l=120, r=20, t=100, b=60),
    )
    return fig


# ── Slot timeline ─────────────────────────────────────────────────────────────
def build_slot_timeline(request, response) -> go.Figure:
    desired = response.desired_slot_result
    feasible_set = {pd.Timestamp(s.slot_time).floor("30min"): s for s in response.feasible_slots}
    desired_key  = pd.Timestamp(desired.slot_time).floor("30min")

    if response.scan_performed:
        as_of = pd.Timestamp(request.as_of_time)
        if as_of.tzinfo is None:
            as_of = as_of.tz_localize(_BOGOTA_TZ)
        day_ts = as_of.normalize()
        if day_ts.tzinfo is None:
            day_ts = day_ts.tz_localize(_BOGOTA_TZ)
        workday_end = day_ts.replace(hour=19, minute=0, second=0)
        rounded_min = math.ceil((as_of.hour * 60 + as_of.minute) / 30) * 30
        scan_start  = day_ts + pd.Timedelta(minutes=rounded_min)
        slot_times, cur = [], scan_start
        while cur <= workday_end:
            slot_times.append(cur)
            cur += pd.Timedelta(minutes=30)
    else:
        slot_times = []

    rows = [{"slot_time": pd.Timestamp(desired.slot_time), "feasible": desired.feasible,
              "reason": desired.reason or "—", "is_desired": True}]
    for t in slot_times:
        key = t.floor("30min")
        if key == desired_key:
            continue
        is_feasible = key in feasible_set
        rows.append({"slot_time": t, "feasible": is_feasible,
                     "reason": "—" if is_feasible else "infeasible", "is_desired": False})

    if not rows:
        return go.Figure().update_layout(title_text="No slots scanned")

    df = pd.DataFrame(rows).sort_values("slot_time").reset_index(drop=True)
    dur_ms = _SLOT_DURATION_MIN * 60 * 1000

    base_vals, x_vals, y_vals, colors, hovers, borders = [], [], [], [], [], []
    for _, row in df.iterrows():
        t = row["slot_time"]
        base_vals.append(_to_plotly_dt(t))
        x_vals.append(dur_ms)
        y_vals.append("slots")
        colors.append(_C_SLOT_FEASIBLE if row["feasible"] else _C_SLOT_INFEASIBLE)
        borders.append("black" if row["is_desired"] else "rgba(0,0,0,0)")
        t_lbl = _to_bogota_naive(t).strftime("%H:%M")
        hovers.append(
            f"{'<b>Desired slot</b><br>' if row['is_desired'] else ''}"
            f"Slot: {t_lbl}<br>Feasible: {row['feasible']}<br>Reason: {row['reason']}"
        )

    fig = go.Figure(go.Bar(
        x=x_vals, y=y_vals, base=base_vals, orientation="h",
        marker=dict(color=colors, line=dict(color=borders, width=2)),
        hovertext=hovers, hoverinfo="text", showlegend=False,
    ))
    for label, color in [("Feasible", _C_SLOT_FEASIBLE), ("Infeasible", _C_SLOT_INFEASIBLE)]:
        fig.add_trace(go.Bar(
            x=[0], y=["slots"], base=[_to_plotly_dt(df["slot_time"].iloc[0])],
            orientation="h", name=label, marker_color=color,
            showlegend=True, hoverinfo="none",
        ))

    _add_vline(fig, desired.slot_time, "crimson", "dash",
               f"Desired {_to_bogota_naive(desired.slot_time).strftime('%H:%M')}",
               annotation_side="right")

    day_str = _to_bogota_naive(desired.slot_time).strftime("%Y-%m-%d")
    feasible_count = int(df["feasible"].sum())
    fig.update_layout(
        title_text=f"Slot Availability Scan — {feasible_count}/{len(df)} feasible",
        barmode="overlay",
        height=200,
        xaxis=_xaxis_config(day_str),
        yaxis=dict(showticklabels=False, title_text=""),
        legend=dict(orientation="h", yanchor="bottom", y=1.1, xanchor="center", x=0.5),
        hovermode="closest",
        margin=dict(l=40, r=20, t=80, b=60),
    )
    return fig


print("Helpers defined.")

Helpers defined.


In [4]:
# ── Interactive scenario selector ─────────────────────────────────────────────
_dropdown = widgets.Dropdown(
    options=list(SCENARIOS.keys()),
    description="Scenario:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
_out = widgets.Output()


def _render(scenario_name):
    state, request, response = _results[scenario_name]
    _out.clear_output(wait=True)
    with _out:
        print("─" * 60)
        display(build_summary_card(request, response))
        print()
        build_gantt_figure(state, request, response).show()
        if response.scan_performed:
            build_slot_timeline(request, response).show()
        else:
            print("✅ Desired slot was feasible — no scan performed.")


_dropdown.observe(lambda c: _render(c["new"]), names="value")
_render(_dropdown.value)
display(_dropdown, _out)

Dropdown(description='Scenario:', layout=Layout(width='420px'), options=('1 — Desired slot feasible (empty sch…

Output()